## Painel Estatístico em Big Data
```
Tema: Impacto da IA e das redes sociais na saúde e no desempenho dos estudantes
```
---

In [ ]:
import time
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ARQUIVO = "estudantes.parquet"

# >>> Se a detecção automática errar, preencha manualmente (ex.: "horas_uso_ia")
COL_IA = None
COL_NOTA = None

inicial = pl.scan_parquet(ARQUIVO)
esquema = inicial.collect_schema()
colunas = esquema.names()
numericas = [c for c, t in esquema.items() if t.is_numeric()]

def candidatas(tokens, entre):
    return [c for c in entre if set(c.split("_")) & tokens]

cand_ia = candidatas({"ia", "ai", "chatgpt", "llm"}, numericas)
cand_nota = candidatas({"nota", "notas", "gpa", "desempenho", "rendimento"}, numericas)

COL_IA = COL_IA or next((c for c in cand_ia if "horas" in c), cand_ia[0] if cand_ia else None)
COL_NOTA = COL_NOTA or (cand_nota[0] if cand_nota else None)

print("Colunas com 'ia' no nome (qualquer tipo):", candidatas({"ia", "ai", "chatgpt", "llm"}, colunas))
print("Coluna de IA em uso   :", COL_IA)
print("Coluna de nota em uso :", COL_NOTA)
if COL_IA is None:
    print("ATENÇÃO: nenhuma coluna numérica de IA encontrada. Veja a lista 'colunas' e preencha COL_IA.")

# Métricas de saúde e desempenho usadas nos resumos
METRICAS = list(dict.fromkeys(
    c for c in ["horas_redes_sociais", "horas_sono", "pontuacao_vicio_digital", "indice_bem_estar",
                "taxa_frequencia_aulas", "capacidade_atencao_minutos", "pontuacao_risco_academico",
                COL_IA, COL_NOTA]
    if c and c in colunas
))

def perfil(lf, grupo):
    """Resumo por grupo: nº de estudantes + média das métricas do tema."""
    return (
        lf.group_by(grupo)
        .agg([pl.len().alias("estudantes")] +
             [pl.col(c).mean().round(2).alias(f"media_{c}") for c in METRICAS])
        .sort(grupo)
    )


def categorizar(lf):
    """Cria categorias de sono, redes sociais e IA (IA em tercis)."""
    exprs = [
        pl.when(pl.col("horas_sono") <= 6).then(pl.lit("1-Baixo (≤6h)"))
          .when(pl.col("horas_sono") < 8).then(pl.lit("2-Médio (6-8h)"))
          .otherwise(pl.lit("3-Bom (≥8h)")).alias("categoria_sono"),
        pl.when(pl.col("horas_redes_sociais") < 2).then(pl.lit("1-Baixo (<2h)"))
          .when(pl.col("horas_redes_sociais") < 5).then(pl.lit("2-Moderado (2-5h)"))
          .otherwise(pl.lit("3-Alto (≥5h)")).alias("categoria_redes"),
    ]
    if COL_IA:
        exprs.append(
            pl.when(pl.col(COL_IA) <= pl.col(COL_IA).quantile(0.33)).then(pl.lit("1-Baixo uso de IA"))
              .when(pl.col(COL_IA) <= pl.col(COL_IA).quantile(0.66)).then(pl.lit("2-Médio uso de IA"))
              .otherwise(pl.lit("3-Alto uso de IA")).alias("categoria_ia")
        )
    return lf.with_columns(exprs)

## Big Data e Processamento Otimizado com Python II
```
PARTE 2
```
Variáveis analisadas: uso de redes sociais, sono, uso de IA e bem-estar (indicador de saúde).

---

In [ ]:
display(inicial.head(5).collect())

### Estatística descritiva, forma, quartis e outliers
```
Função única que calcula: média, mediana, moda, variância, desvio padrão, amplitude, quartis, assimetria, curtose e outliers (IQR).
```
---

In [ ]:
def analisar_variavel(col, rotulo):
    print(f"\n{'=' * 60}\n{rotulo} ({col})\n{'=' * 60}")

    m = inicial.select(
        pl.col(col).mean().alias("Média"),
        pl.col(col).median().alias("Mediana"),
        pl.col(col).mode().first().alias("Moda"),
        pl.col(col).var().alias("Variância"),
        pl.col(col).std().alias("Desvio_Padrão"),
        (pl.col(col).max() - pl.col(col).min()).alias("Amplitude"),
        pl.col(col).quantile(0.25).alias("Q1"),
        pl.col(col).quantile(0.75).alias("Q3"),
        pl.col(col).skew().alias("Assimetria"),
        pl.col(col).kurtosis().alias("Curtose"),
    ).collect()
    display(m)

    # Assimetria
    a = m["Assimetria"][0]
    if -0.5 <= a <= 0.5:
        print(f"Assimetria ({a:.2f}): distribuição simétrica.")
    elif a > 0.5:
        print(f"Assimetria ({a:.2f}): positiva (cauda à direita, valores altos puxam a média).")
    else:
        print(f"Assimetria ({a:.2f}): negativa (cauda à esquerda).")

    # Outliers (IQR)
    Q1, Q3 = m["Q1"][0], m["Q3"][0]
    iqr = Q3 - Q1
    lim_inf, lim_sup = Q1 - 1.5 * iqr, Q3 + 1.5 * iqr
    outliers = inicial.filter(
        (pl.col(col) < lim_inf) | (pl.col(col) > lim_sup)
    ).select(["pais", "id_estudante", col]).collect()
    total = inicial.select(pl.len()).collect().item()
    print(f"Limites IQR: [{lim_inf:.2f}, {lim_sup:.2f}] | Outliers: {outliers.height:,} ({outliers.height/total:.2%})")
    if outliers.height:
        display(outliers.sort(col, descending=True).head(5))

    # Gráficos
    valores = inicial.select(col).collect()[col].to_numpy()
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(valores, bins=30, kde=True, color="steelblue", ax=ax[0])
    ax[0].set_title(f"Distribuição: {rotulo}")
    ax[0].set_xlabel(col)
    sns.boxplot(x=valores, color="mediumpurple", ax=ax[1])
    ax[1].set_title(f"Boxplot: {rotulo}")
    ax[1].set_xlabel(col)
    plt.tight_layout()
    plt.show()

In [ ]:
analisar_variavel("horas_redes_sociais", "Horas em redes sociais")

In [ ]:
if COL_IA:
    analisar_variavel(COL_IA, "Uso de IA")

In [ ]:
analisar_variavel("horas_sono", "Horas de sono")

In [ ]:
analisar_variavel("indice_bem_estar", "Índice de bem-estar")

### Comparação entre grupos
```
Distribuição do bem-estar e da frequência por nível de uso de redes sociais (e de IA).
```
---

In [ ]:
lf_cat = categorizar(inicial)
grupos = ["categoria_redes"] + (["categoria_ia"] if COL_IA else [])
alvos = [c for c in ["indice_bem_estar", "taxa_frequencia_aulas"] if c in colunas]

amostra = lf_cat.select(grupos + alvos).collect().sample(n=50_000, seed=42).to_pandas()

fig, axes = plt.subplots(len(alvos), len(grupos), figsize=(6 * len(grupos), 4.5 * len(alvos)), squeeze=False)
for i, alvo in enumerate(alvos):
    for j, g in enumerate(grupos):
        ordem = sorted(amostra[g].unique())
        sns.boxplot(data=amostra, x=g, y=alvo, order=ordem, ax=axes[i][j], color="lightsteelblue")
        axes[i][j].set_title(f"{alvo} por {g}")
        axes[i][j].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

### Arquivos Parquet processados (entrega)
```
Gera os Parquet processados: base completa com categorias + resumos analíticos.
```
---

In [ ]:
import os
os.makedirs("parquet", exist_ok=True)

lf_cat = categorizar(inicial)

# 1) Base completa com as colunas derivadas (categorias)
lf_cat.collect().write_parquet("parquet/estudantes_processado.parquet")

# 2) Resumo por país
perfil(lf_cat, "pais").collect().write_parquet("parquet/resumo_por_pais.parquet")

# 3) Resumo redes sociais x sono
(
    lf_cat.group_by(["categoria_redes", "categoria_sono"])
    .agg(
        pl.len().alias("estudantes"),
        pl.col("indice_bem_estar").mean().round(2).alias("media_bem_estar"),
        pl.col("taxa_frequencia_aulas").mean().round(2).alias("media_frequencia"),
    )
    .sort(["categoria_redes", "categoria_sono"])
    .collect()
    .write_parquet("parquet/resumo_redes_x_sono.parquet")
)

for f in sorted(os.listdir("parquet")):
    print(f"{f:45s} {os.path.getsize(os.path.join('parquet', f)) / 1e6:8.2f} MB")

# (Opcional) conversão do CSV original -> Parquet, se precisar entregar essa etapa:
# pl.scan_csv("arquivo_original.csv").sink_parquet("parquet/estudantes_convertido.parquet")

### Interpretação (complete com seus números)
- **Redes sociais:** média, mediana e dispersão; a distribuição é simétrica ou assimétrica? Quantos outliers (heavy users)?
- **IA:** como é o padrão de uso comparado às redes?
- **Grupos:** o bem-estar e a frequência diminuem conforme sobe o uso de redes/IA? Qual a diferença entre o grupo de menor e o de maior uso?
- **Cuidado:** diferença entre grupos descreve associação, não prova que o uso *causa* a diferença.